# Welcome to Colab!

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision import datasets
import numpy as np
import os

# ─── 1. Kaggle setup ───────────────────────────────────────────
from google.colab import files
files.upload()

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
print("Kaggle API key configured!")

# ─── 2. Download dataset ───────────────────────────────────────
os.system("kaggle datasets download -d msambare/fer2013")
os.system("unzip -q fer2013.zip")
print("Dataset downloaded!")

# ─── 3. Data loaders ───────────────────────────────────────────
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = datasets.ImageFolder(root="/content/train", transform=transform)
test_dataset  = datasets.ImageFolder(root="/content/test",  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))
print("Classes:", train_dataset.classes)

# ─── 4. Model ──────────────────────────────────────────────────
class EmotionCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1   = nn.Conv2d(1, 32,  kernel_size=3, padding=1)
        self.conv2   = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3   = nn.Conv2d(64, 128,kernel_size=3, padding=1)
        self.pool    = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.25)
        self.fc1     = nn.Linear(128 * 6 * 6, 256)
        self.fc2     = nn.Linear(256, 7)
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.pool(self.relu(self.conv3(x)))
        x = self.dropout(x)
        x = x.view(-1, 128 * 6 * 6)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

device    = torch.device("cpu")
model     = EmotionCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print("Model ready! Using device:", device)

# ─── 5. Training ───────────────────────────────────────────────
for epoch in range(10):
    model.train()
    running_loss = correct = total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted  = torch.max(outputs, 1)
        total        += labels.size(0)
        correct      += (predicted == labels).sum().item()

    print(f"Epoch [{epoch+1}/10] Loss: {running_loss/len(train_loader):.4f}  Accuracy: {100*correct/total:.2f}%")

print("Training complete!")

# ─── 6. Evaluation ─────────────────────────────────────────────
model.eval()
correct = total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs        = model(images)
        _, predicted   = torch.max(outputs, 1)
        total         += labels.size(0)
        correct       += (predicted == labels).sum().item()

print(f"Test Accuracy: {100*correct/total:.2f}%")

# ─── 7. Predict a single sample ────────────────────────────────
emotion_labels = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']

model.eval()
sample_images, sample_labels = next(iter(test_loader))

with torch.no_grad():
    outputs    = model(sample_images.to(device))
    _, preds   = torch.max(outputs, 1)

for i in range(5):
    actual    = emotion_labels[sample_labels[i].item()]
    predicted = emotion_labels[preds[i].item()]
    print(f"Sample {i+1} | Actual: {actual:10s} | Predicted: {predicted}")

Saving kaggle.json to kaggle.json
Kaggle API key configured!
Dataset downloaded!
Train samples: 28709
Test samples: 7178
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Model ready! Using device: cpu
Epoch [1/10] Loss: 1.6031  Accuracy: 36.93%
Epoch [2/10] Loss: 1.3543  Accuracy: 48.40%
Epoch [3/10] Loss: 1.2390  Accuracy: 52.42%
Epoch [4/10] Loss: 1.1468  Accuracy: 56.41%
Epoch [5/10] Loss: 1.0757  Accuracy: 59.22%
Epoch [6/10] Loss: 1.0018  Accuracy: 62.34%
Epoch [7/10] Loss: 0.9322  Accuracy: 65.20%
Epoch [8/10] Loss: 0.8651  Accuracy: 67.42%
Epoch [9/10] Loss: 0.8040  Accuracy: 70.01%
Epoch [10/10] Loss: 0.7370  Accuracy: 72.50%
Training complete!
Test Accuracy: 59.13%
Sample 1 | Actual: angry      | Predicted: angry
Sample 2 | Actual: angry      | Predicted: angry
Sample 3 | Actual: angry      | Predicted: sad
Sample 4 | Actual: angry      | Predicted: sad
Sample 5 | Actual: angry      | Predicted: angry
